# 제조 음성 ASR 심사 파이프라인

하나의 노트북에서 세 가지 데이터 모드를 사용합니다.

- `PUBLIC_PROXY`: 공개 Zeroth 한국어 음성으로 데이터 준비부터 모델 비교, 자동 프록시 선택, LoRA, 양자화, 고정 Test, 보고서·백데이터까지 전체 동작을 검증합니다.
- `SYNTHETIC_MANUFACTURING`: 저장소의 한국어 TTS 제조 문장 30개로 실제 제조 데이터를 넣기 전 동일한 코드 경로를 검증합니다.
- `PRIVATE_MANUFACTURING`: 나중에 승인된 제조 녹음과 검수 전사로 같은 코드를 다시 실행합니다. 이 모드의 모델·양자화 선택은 반드시 사람이 수행합니다.

> 공개·합성 결과는 코드와 산출물의 정상 동작 증거입니다. 제조 현장 성능, 배포 적합성 또는 심사 최종 결론의 증거로 사용하면 안 됩니다.

**보안:** 카메라·마이크·패스키를 사용하지 않습니다. 실제 음성은 GitHub에 올리지 않고 승인된 비공개 Drive 경로만 사용합니다.

## 0. 실행 모드

In [ ]:
# "PUBLIC_PROXY", "SYNTHETIC_MANUFACTURING", "PRIVATE_MANUFACTURING"
# 현재 실행을 합성 제조 TTS 데이터 모드로 선택합니다. 실제 데이터 사용 시 값을 바꿉니다.
DATA_MODE = "SYNTHETIC_MANUFACTURING"

# Colab에서 복제할 ASR 프로젝트 GitHub 저장소 주소를 지정합니다.
GITHUB_REPO_URL = "https://github.com/Pronesis9758/aias-specialist-asr.git"
# 벤치마크·양자화 기능이 있는 작업 브랜치를 선택합니다. PR 병합 후 main으로 바꿉니다.
GITHUB_BRANCH = "codex/whisper-benchmark-quantization"  # PR 병합 후 main
# GitHub 코드를 복제할 Colab 임시 경로입니다. 런타임 종료·초기화 시 삭제됩니다.
PROJECT_DIR = "/content/AIAS"
# 모델·결과·보고서를 보존할 내 Google Drive 경로입니다. 런타임 종료 후에도 유지됩니다.
DRIVE_ROOT = "/content/drive/MyDrive/AI_Specialist_ASR_Project"

# MODE_SETTINGS 변수에 이 연구 단계에서 계산하거나 선택한 값을 저장합니다.
MODE_SETTINGS = {
    # 공개 Zeroth 한국어 음성으로 전체 파이프라인만 검증하는 프록시 모드를 정의합니다.
    "PUBLIC_PROXY": {
        # 공개 Zeroth 데이터·평가·Drive 산출물 경로가 담긴 설정 파일을 연결합니다.
        "config": "configs/public_proxy_assessment.yaml",
        # 공개 프록시에서 비교할 Whisper 후보와 실행 조건 목록을 지정합니다.
        "matrix": "configs/benchmarks/public_proxy_whisper_models.yaml",
        # 공개 프록시 모델에 적용할 float16·int8 양자화 비교 조건을 지정합니다.
        "quantization": "configs/quantization/public_proxy_whisper_quantization.yaml",
        # 공개 프록시 모델 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "benchmark_id": "public-proxy-whisper-model-benchmark-v1",
        # 공개 프록시 양자화 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "quantization_id": "public-proxy-whisper-quantization-v1",
    # 앞에서 시작한 코드 구문을 닫습니다.
    },
    # 제조 용어가 포함된 합성 TTS 30개로 기능을 검증하는 기본 실행 모드를 정의합니다.
    "SYNTHETIC_MANUFACTURING": {
        # 합성 제조 데이터 분할·학습·평가 조건이 담긴 설정 파일을 연결합니다.
        "config": "configs/synthetic_manufacturing_sample.yaml",
        # 합성 제조 데이터에서 비교할 tiny·base·small 후보 목록을 지정합니다.
        "matrix": "configs/benchmarks/synthetic_manufacturing_whisper_models.yaml",
        # 합성 제조 선택 모델의 float16·int8-float16 비교 조건을 지정합니다.
        "quantization": "configs/quantization/synthetic_manufacturing_whisper_quantization.yaml",
        # 합성 제조 모델 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "benchmark_id": "synthetic-manufacturing-whisper-model-benchmark-v1",
        # 합성 제조 양자화 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "quantization_id": "synthetic-manufacturing-whisper-quantization-v1",
    # 앞에서 시작한 코드 구문을 닫습니다.
    },
    # 승인된 실제 제조 녹음과 사람 검수 전사를 사용하는 최종 연구 모드를 정의합니다.
    "PRIVATE_MANUFACTURING": {
        # 실제 제조 데이터 경로와 엄격한 거버넌스 조건을 입력할 템플릿을 연결합니다.
        "config": "configs/manufacturing_private_template.yaml",
        # 실제 제조 데이터에서 사람이 검토할 확장 Whisper 후보 목록을 지정합니다.
        "matrix": "configs/benchmarks/manufacturing_whisper_models_template.yaml",
        # 실제 제조 선택 모델에 적용할 양자화 후보와 허용 손실 조건을 지정합니다.
        "quantization": "configs/quantization/manufacturing_whisper_quantization_template.yaml",
        # 실제 제조 모델 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "benchmark_id": "manufacturing-whisper-model-benchmark-v1",
        # 실제 제조 양자화 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "quantization_id": "manufacturing-whisper-quantization-v1",
    # 앞에서 시작한 코드 구문을 닫습니다.
    },
# 앞에서 시작한 코드 구문을 닫습니다.
}
# 지원 목록에 없는 데이터 모드를 조기에 차단합니다.
if DATA_MODE not in MODE_SETTINGS:
    # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
    raise ValueError(f"지원하지 않는 DATA_MODE: {DATA_MODE}")

# 선택한 데이터 모드의 설정 묶음만 꺼냅니다.
mode = MODE_SETTINGS[DATA_MODE]
# 선택 모드의 데이터·학습·평가 설정 파일 경로를 사용합니다.
CONFIG = mode["config"]
# 선택 모드에서 비교할 Whisper 후보 목록 경로를 사용합니다.
MODEL_MATRIX = mode["matrix"]
# 선택 모델에 적용할 양자화 후보와 평가 조건 파일 경로를 사용합니다.
QUANTIZATION_SPEC = mode["quantization"]
# 모델 비교 결과를 저장할 실험 그룹 ID를 사용합니다.
BENCHMARK_ID = mode["benchmark_id"]
# 양자화 비교 결과를 저장할 실험 그룹 ID를 사용합니다.
QUANTIZATION_ID = mode["quantization_id"]
# 실제 제조 음성·정답·승인 문서를 둘 비공개 Drive 폴더를 지정합니다.
PRIVATE_ROOT = f"{DRIVE_ROOT}/data/private/manufacturing"
# 현재 실행이 공개 데이터 기능 검증 모드인지 표시합니다.
IS_PUBLIC_PROXY = DATA_MODE == "PUBLIC_PROXY"
# 현재 실행이 합성 제조 데이터 기능 검증 모드인지 표시합니다.
IS_SYNTHETIC_MANUFACTURING = DATA_MODE == "SYNTHETIC_MANUFACTURING"
# 공개·합성 모드에서는 사람 결정 대신 자동 선택을 허용하도록 표시합니다.
IS_AUTOMATED_PROXY = IS_PUBLIC_PROXY or IS_SYNTHETIC_MANUFACTURING
# 사용자가 확인할 수 있도록 현재 데이터 모드를 출력합니다.
print("Mode:", DATA_MODE)
# 현재 모드가 사용하는 핵심 YAML 설정 경로를 출력합니다.
print("Config:", CONFIG)

## 1. GPU와 Google Drive 연결

In [ ]:
# 할당된 GPU 종류와 메모리 상태를 확인합니다.
!nvidia-smi
# google.colab 모듈에서 필요한 기능을 불러옵니다.
from google.colab import drive

# 모델 캐시와 실험 산출물을 보존할 Google Drive를 연결합니다.
drive.mount("/content/drive")

## 2. GitHub 코드 동기화

In [ ]:
# os 모듈을 불러옵니다.
import os
# subprocess 모듈을 불러옵니다.
import subprocess

# Colab 임시 디스크에 저장소가 아직 없는지 확인합니다.
if not os.path.exists(PROJECT_DIR):
    # 외부 명령을 실행하고 실패하면 즉시 예외를 발생시킵니다.
    subprocess.run(
        # 여러 값으로 구성된 자료 구조를 시작합니다.
        [
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "git",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "clone",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "--branch",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            GITHUB_BRANCH,
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "--single-branch",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            GITHUB_REPO_URL,
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            PROJECT_DIR,
        # 앞에서 시작한 코드 구문을 닫습니다.
        ],
        # check 변수에 이 연구 단계에서 계산하거나 선택한 값을 저장합니다.
        check=True,
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
# 앞선 조건에 해당하지 않는 경우를 처리합니다.
else:
    # 외부 명령을 실행하고 실패하면 즉시 예외를 발생시킵니다.
    subprocess.run(
        # 여러 값으로 구성된 자료 구조를 시작합니다.
        ["git", "-C", PROJECT_DIR, "fetch", "origin", GITHUB_BRANCH],
        # check 변수에 이 연구 단계에서 계산하거나 선택한 값을 저장합니다.
        check=True,
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
    # 외부 명령을 실행하고 실패하면 즉시 예외를 발생시킵니다.
    subprocess.run(
        # 여러 값으로 구성된 자료 구조를 시작합니다.
        ["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH],
        # check 변수에 이 연구 단계에서 계산하거나 선택한 값을 저장합니다.
        check=True,
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
    # 외부 명령을 실행하고 실패하면 즉시 예외를 발생시킵니다.
    subprocess.run(
        # 여러 값으로 구성된 자료 구조를 시작합니다.
        [
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "git",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "-C",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            PROJECT_DIR,
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "merge",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            "--ff-only",
            # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
            f"origin/{GITHUB_BRANCH}",
        # 앞에서 시작한 코드 구문을 닫습니다.
        ],
        # check 변수에 이 연구 단계에서 계산하거나 선택한 값을 저장합니다.
        check=True,
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
# 이후 상대 경로가 저장소를 기준으로 동작하도록 작업 폴더를 바꿉니다.
os.chdir(PROJECT_DIR)
# 재현성을 위해 실제 실행 중인 Git 커밋 해시를 출력합니다.
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. 의존성 설치와 실행 함수

In [ ]:
# Colab 기본 패키지 중 충돌 가능성이 있는 항목을 제거합니다.
%pip uninstall -y torchao gradio gradio-client
# 프로젝트와 학습용 의존성을 현재 Colab 런타임에 설치합니다.
%pip install -q -e ".[train]" "transformers>=4.46,<5" "peft>=0.14,<0.19"

# importlib.util 모듈에서 필요한 기능을 불러옵니다.
from importlib.util import find_spec
# sys 모듈을 불러옵니다.
import sys

# editable 설치 직후 프로젝트 패키지를 찾을 src 절대 경로를 계산합니다.
PROJECT_SRC = os.path.join(PROJECT_DIR, "src")
# 프로젝트 src 경로가 Python 검색 경로에 없는지 확인합니다.
if PROJECT_SRC not in sys.path:
    # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
    sys.path.insert(0, PROJECT_SRC)
# AIAS 패키지를 실제로 import할 수 있는지 확인합니다.
if find_spec("aias_specialist") is None:
    # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
    raise ModuleNotFoundError("aias_specialist import path refresh failed")
# 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
print("aias_specialist import OK:", PROJECT_SRC)


# run_aias 재사용 함수를 정의합니다.
def run_aias(*args):
    # 현재 Python으로 AIAS CLI와 전달받은 세부 명령을 실행할 명령 배열을 만듭니다.
    command = [sys.executable, "-m", "aias_specialist.cli", *args]
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("\nRunning:", " ".join(command), flush=True)
    # 학습·평가 진행 로그가 지연 없이 Colab에 표시되도록 실행 환경을 만듭니다.
    environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
    # 외부 명령을 실행하고 실패하면 즉시 예외를 발생시킵니다.
    subprocess.run(command, check=True, env=environment)

## 4. 데이터 준비

`PUBLIC_PROXY`에서는 고정 revision의 `kresnik/zeroth_korean` 일부만 스트리밍해 Drive에 저장합니다. `SYNTHETIC_MANUFACTURING`에서는 저장소에 포함된 TTS WAV와 정답 manifest를 검증합니다. `PRIVATE_MANUFACTURING`에서는 기존 파일을 덮어쓰지 않고 입력 양식을 준비합니다.

In [ ]:
# pathlib 모듈에서 필요한 기능을 불러옵니다.
from pathlib import Path
# json 모듈을 불러옵니다.
import json
# shutil 모듈을 불러옵니다.
import shutil
# pandas 모듈을 불러옵니다.
import pandas as pd

# 공개 Zeroth 프록시 모드일 때의 데이터·선택 절차를 실행합니다.
if IS_PUBLIC_PROXY:
    # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
    run_aias("prepare-hf-dataset", "--config", CONFIG)
    # 다운로드한 Zeroth 음성과 manifest를 보존할 Drive 폴더를 지정합니다.
    public_root = Path(DRIVE_ROOT) / "data/public/zeroth_korean"
    # 결과를 Colab 표 형태로 표시합니다.
    display(pd.read_csv(public_root / "manifest.csv").groupby("split").size())
    # 현재 데이터셋의 출처·생성 방식·라이선스 정보를 담은 JSON 경로를 지정합니다.
    provenance_path = public_root / "dataset_provenance.json"
    # 데이터 출처·라이선스 JSON이 생성됐는지 확인합니다.
    if provenance_path.exists():
        # 결과를 Colab 표 형태로 표시합니다.
        display(json.loads(provenance_path.read_text(encoding="utf-8")))
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("PUBLIC PROXY: 제조 성능 증거가 아닌 전체동작 검증 데이터입니다.")
# 합성 제조 TTS 모드일 때 manifest와 출처를 검증합니다.
elif IS_SYNTHETIC_MANUFACTURING:
    # aias_specialist.config 모듈에서 필요한 기능을 불러옵니다.
    from aias_specialist.config import load_settings
    # aias_specialist.data 모듈에서 필요한 기능을 불러옵니다.
    from aias_specialist.data import validate_manifest

    # 합성 제조 manifest와 실험 경로를 YAML 설정에서 읽습니다.
    synthetic_settings = load_settings(CONFIG)
    # 합성 WAV 존재 여부·정답·split·해시를 검증한 manifest를 저장합니다.
    synthetic_manifest = validate_manifest(
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        synthetic_settings.paths.manifest, backend="faster_whisper"
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
    # 결과를 Colab 표 형태로 표시합니다.
    display(synthetic_manifest.groupby(["split", "noise_condition"]).size())
    # 현재 데이터셋의 출처·생성 방식·라이선스 정보를 담은 JSON 경로를 지정합니다.
    provenance_path = synthetic_settings.paths.manifest.parent / "dataset_provenance.json"
    # 결과를 Colab 표 형태로 표시합니다.
    display(json.loads(provenance_path.read_text(encoding="utf-8")))
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("SYNTHETIC: 실제 제조 성능이나 사람 검수 증거가 아닙니다.")
# 앞선 조건에 해당하지 않는 경우를 처리합니다.
else:
    # 실제 제조 데이터의 비공개 Drive 문자열 경로를 Path 객체로 변환합니다.
    private_root = Path(PRIVATE_ROOT)
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    (private_root / "audio").mkdir(parents=True, exist_ok=True)
    # 실제 제조 모드에서 필요한 manifest·승인·검토·합격기준 양식을 연결합니다.
    templates = {
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        Path("data/templates/manufacturing_manifest_template.csv"): private_root / "manifest.csv",
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        Path("data/templates/data_approval_template.yaml"): private_root / "data_approval.yaml",
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        Path("data/templates/human_review_signoff_template.yaml"): private_root
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        / "human_review_signoff.yaml",
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        Path("configs/assessment/acceptance_criteria_template.yaml"): private_root
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        / "acceptance_criteria.yaml",
    # 앞에서 시작한 코드 구문을 닫습니다.
    }
    # 각 항목을 순회하며 같은 처리를 반복합니다.
    for source, destination in templates.items():
        # 기존 비공개 입력·검토 문서를 덮어쓰지 않도록 확인합니다.
        if not destination.exists():
            # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
            shutil.copy2(source, destination)
            # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
            print("Created:", destination)
        # 앞선 조건에 해당하지 않는 경우를 처리합니다.
        else:
            # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
            print("Preserved existing:", destination)
    # 문서·데이터·실험·거버넌스·사람 검토 증거를 종합 점검합니다.
    run_aias(
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        "assessment-audit",
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        "--config",
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        CONFIG,
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        "--output-dir",
        # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
        f"{DRIVE_ROOT}/reports/assessment_readiness/private_manufacturing",
    # 앞에서 시작한 코드 구문을 닫습니다.
    )

## 5. Whisper 모델 비교

공개·합성 모드는 빠른 전체동작 검증을 위해 `tiny`, `base`, `small`을 비교합니다. 실제 제조 모드는 6개 후보를 비교합니다. 모델과 변환본은 Drive 캐시에 재사용됩니다.

In [ ]:
# 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
run_aias("model-matrix-lock", "--matrix", MODEL_MATRIX)
# 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
run_aias("benchmark-models", "--matrix", MODEL_MATRIX)
# 모델별 예측·성능표·보고서가 저장된 Drive 벤치마크 폴더를 지정합니다.
benchmark_dir = Path(DRIVE_ROOT) / "artifacts/benchmarks" / BENCHMARK_ID
# Whisper 후보별 정확도·속도·메모리·용량 비교표를 읽습니다.
benchmark_table = pd.read_csv(benchmark_dir / "benchmark_comparison.csv")
# 결과를 Colab 표 형태로 표시합니다.
display(benchmark_table)

## 6. 모델 선택

공개·합성 모드는 완료 후보 중 rank 1을 자동 선택하지만 사람 검토로 기록하지 않습니다. 실제 제조 모드에서는 아래 사람 검토 값을 직접 입력해야 다음 단계로 진행됩니다.

In [ ]:
# best_completed_member 재사용 함수를 정의합니다.
def best_completed_member(frame):
    # 실패 후보를 제외하고 평가가 완료된 모델 또는 양자화 후보만 남깁니다.
    completed = frame.loc[frame["status"].eq("completed")].copy()
    # 정상 완료된 비교 후보가 하나도 없는지 확인합니다.
    if completed.empty:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise RuntimeError("완료된 후보가 없습니다. 위 오류를 먼저 확인하세요.")
    # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
    completed["rank"] = pd.to_numeric(completed["rank"], errors="coerce")
    # 계산하거나 선택한 결과를 호출한 곳에 반환합니다.
    return completed.sort_values(
        # 여러 값으로 구성된 자료 구조를 시작합니다.
        ["rank", "cer", "wer", "aggregate_real_time_factor"],
        # 순위 정렬 시 결측 성능값을 가장 뒤로 보내도록 지정합니다.
        na_position="last",
    # 앞에서 시작한 코드 구문을 닫습니다.
    ).iloc[0]


# 공개·합성 기능 검증 모드이면 순위 기반 자동 선택을 사용합니다.
if IS_AUTOMATED_PROXY:
    # 완료된 Whisper 후보 중 종합 순위가 가장 높은 행을 선택합니다.
    selected_row = best_completed_member(benchmark_table)
    # 자동 선택된 Whisper 후보의 member_id를 후속 학습 모델로 사용합니다.
    SELECTED_MODEL = str(selected_row["member_id"])
    # 공개 Zeroth 프록시 모드일 때의 데이터·선택 절차를 실행합니다.
    if IS_PUBLIC_PROXY:
        # 공개 프록시의 자동 선택임을 기록해 사람 검토와 구분합니다.
        REVIEWER = "AUTOMATED_PUBLIC_PROXY"
        # 모델 선택의 데이터 범위·정확도·속도·검토 한계를 근거로 기록합니다.
        MODEL_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "공개 Zeroth 프록시 rank 1 자동 선택. 코드·산출물 smoke test "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "전용이며 제조 모델 선정 또는 사람 검토 증거가 아님."
        # 앞에서 시작한 코드 구문을 닫습니다.
        )
    # 앞선 조건에 해당하지 않는 경우를 처리합니다.
    else:
        # 합성 데이터의 자동 선택임을 기록해 실제 제조 검토와 구분합니다.
        REVIEWER = "AUTOMATED_SYNTHETIC_FIXTURE"
        # 모델 선택의 데이터 범위·정확도·속도·검토 한계를 근거로 기록합니다.
        MODEL_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "합성 제조 TTS rank 1 자동 선택. 기능 검증 전용이며 실제 제조 "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "모델 선정 또는 사람 검토 증거가 아님."
        # 앞에서 시작한 코드 구문을 닫습니다.
        )
    # 자동 선택 결과를 사람 검토로 오인하지 않도록 프록시 표시 인자를 추가합니다.
    extra_selection_args = ["--automated-proxy"]
# 앞선 조건에 해당하지 않는 경우를 처리합니다.
else:
    # 실제 제조 모드에서 비교표 검토 후 선택할 모델의 초기 입력값입니다.
    SELECTED_MODEL = "small"  # 비교표를 보고 수정
    # 실제 제조 모델을 검토한 담당자 이름을 반드시 입력해야 하는 자리입니다.
    REVIEWER = "TO_BE_COMPLETED"
    # 모델 선택의 데이터 범위·정확도·속도·검토 한계를 근거로 기록합니다.
    MODEL_REASON = "TO_BE_COMPLETED: 정확도·속도·메모리·거버넌스 근거"
    # 실제 제조 모델의 검토자 또는 선택 근거가 미입력 상태인지 확인합니다.
    if "TO_BE_COMPLETED" in REVIEWER or "TO_BE_COMPLETED" in MODEL_REASON:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise ValueError("제조 모드에서는 사람 검토자와 모델 선택 근거를 입력하세요.")
    # 실제 제조 모드에서는 자동 프록시 표시 없이 사람 검토 기록을 사용합니다.
    extra_selection_args = []

# 검토한 모델과 선택자·근거를 변경 불가능한 선택 기록으로 남깁니다.
run_aias(
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "select-model",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--benchmark-dir",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    str(benchmark_dir),
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--model-id",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    SELECTED_MODEL,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--reviewer",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    REVIEWER,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--reason",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    MODEL_REASON,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    *extra_selection_args,
# 앞에서 시작한 코드 구문을 닫습니다.
)
# 선택 모델·revision·성능·선택 사유가 기록된 YAML 경로를 지정합니다.
model_selection = benchmark_dir / "model_selection.yaml"
# 자동 또는 사람 검토로 선택된 Whisper 모델 ID를 출력합니다.
print("Selected model:", SELECTED_MODEL)
# 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
print(model_selection.read_text(encoding="utf-8"))

## 7. 선택 모델 LoRA

공개·합성 모드는 작은 train split과 30 step의 짧은 실행으로 학습 코드, checkpoint, Base/LoRA 비교 산출물을 검증합니다. 실제 제조 모드는 500 step을 사용하며 데이터 규모에 맞춰 조정합니다.

In [ ]:
# 선택된 Whisper에 LoRA를 학습하고 Base 대비 결과를 생성합니다.
run_aias(
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "train-selected-whisper",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--selection",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    str(model_selection),
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--config",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    CONFIG,
# 앞에서 시작한 코드 구문을 닫습니다.
)

## 8. 양자화 비교

In [ ]:
# 선택 모델의 각 양자화 variant를 동일 조건에서 평가합니다.
run_aias(
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "quantization-sweep",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--spec",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    QUANTIZATION_SPEC,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--selection",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    str(model_selection),
# 앞에서 시작한 코드 구문을 닫습니다.
)
# 양자화별 예측·성능표·보고서가 저장된 Drive 폴더를 지정합니다.
quantization_dir = Path(DRIVE_ROOT) / "artifacts/quantization" / QUANTIZATION_ID
# 양자화 후보별 정확도 변화·속도·메모리·용량 비교표를 읽습니다.
quantization_table = pd.read_csv(quantization_dir / "quantization_comparison.csv")
# 결과를 Colab 표 형태로 표시합니다.
display(quantization_table)

## 9. 양자화 선택

공개·합성 모드는 종합 rank 1을 자동 선택합니다. 실제 제조 모드는 정확도 손실, RTF, GPU 메모리와 모델 용량을 사람이 함께 검토합니다.

In [ ]:
# 공개·합성 기능 검증 모드이면 순위 기반 자동 선택을 사용합니다.
if IS_AUTOMATED_PROXY:
    # 완료된 양자화 후보 중 종합 순위가 가장 높은 행을 선택합니다.
    selected_quantization_row = best_completed_member(quantization_table)
    # 자동 선택된 양자화 variant ID를 최종 Test 평가에 사용합니다.
    SELECTED_VARIANT = str(selected_quantization_row["member_id"])
    # 공개 Zeroth 프록시 모드일 때의 데이터·선택 절차를 실행합니다.
    if IS_PUBLIC_PROXY:
        # 양자화 선택의 정확도 손실·속도·메모리·용량 근거를 기록합니다.
        QUANTIZATION_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "공개 Zeroth 프록시 종합 rank 1 자동 선택. 양자화 코드·산출물 "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "smoke test 전용이며 실제 배포 결정 또는 사람 검토 증거가 아님."
        # 앞에서 시작한 코드 구문을 닫습니다.
        )
    # 앞선 조건에 해당하지 않는 경우를 처리합니다.
    else:
        # 양자화 선택의 정확도 손실·속도·메모리·용량 근거를 기록합니다.
        QUANTIZATION_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "합성 제조 TTS 종합 rank 1 자동 선택. 기능 검증 전용이며 실제 "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "배포 결정 또는 사람 검토 증거가 아님."
        # 앞에서 시작한 코드 구문을 닫습니다.
        )
    # 양자화 자동 선택을 사람 배포 결정으로 오인하지 않도록 프록시 표시를 추가합니다.
    extra_quantization_args = ["--automated-proxy"]
# 앞선 조건에 해당하지 않는 경우를 처리합니다.
else:
    # 실제 제조 모드에서 손실·속도·메모리를 검토한 뒤 선택할 초기 양자화 값입니다.
    SELECTED_VARIANT = "float16"  # 비교표를 보고 수정
    # 양자화 선택의 정확도 손실·속도·메모리·용량 근거를 기록합니다.
    QUANTIZATION_REASON = "TO_BE_COMPLETED: 정확도 손실·속도·메모리·모델 용량 근거"
    # 실제 제조 양자화 선택 근거가 미입력 상태인지 확인합니다.
    if "TO_BE_COMPLETED" in QUANTIZATION_REASON:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise ValueError("제조 모드에서는 양자화 선택 근거를 입력하세요.")
    # 실제 제조 모드에서는 자동 프록시 표시 없이 사람의 양자화 결정을 기록합니다.
    extra_quantization_args = []

# 검토한 양자화와 선택자·근거를 선택 기록으로 남깁니다.
run_aias(
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "select-quantization",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--quantization-dir",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    str(quantization_dir),
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--variant-id",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    SELECTED_VARIANT,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--reviewer",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    REVIEWER,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--reason",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    QUANTIZATION_REASON,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    *extra_quantization_args,
# 앞에서 시작한 코드 구문을 닫습니다.
)
# 최종 variant·성능·선택 사유가 기록된 YAML 경로를 지정합니다.
quantization_selection = quantization_dir / "quantization_selection.yaml"
# 자동 또는 사람 검토로 선택된 양자화 variant ID를 출력합니다.
print("Selected variant:", SELECTED_VARIANT)
# 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
print(quantization_selection.read_text(encoding="utf-8"))

## 10. 고정 Test 최종평가

모델·양자화 선택이 끝난 뒤에만 그동안 보지 않은 Test split을 한 번 평가합니다.

In [ ]:
# 선택이 끝난 모델을 미사용 고정 Test split에서 한 번 평가합니다.
run_aias(
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "finalize-evaluation",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--selection",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    str(quantization_selection),
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--config",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    CONFIG,
# 앞에서 시작한 코드 구문을 닫습니다.
)

## 11. 산출물·심사 준비도 확인

공개·합성 모드에서는 `not_ready`가 정상입니다. 공개·합성 데이터, 자동 선택, 미완료 사람 서명은 제조 심사 증거를 대체하지 못합니다.

In [ ]:
# 현재 데이터 모드의 심사 준비도 JSON·Markdown을 저장할 Drive 폴더를 지정합니다.
readiness_dir = Path(DRIVE_ROOT) / "reports/assessment_readiness" / DATA_MODE.lower()
# 문서·데이터·실험·거버넌스·사람 검토 증거를 종합 점검합니다.
run_aias(
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "assessment-audit",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--config",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    CONFIG,
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    "--output-dir",
    # 바로 위 함수·목록·사전 구문에 이 줄의 구체적인 값 또는 인자를 전달합니다.
    str(readiness_dir),
# 앞에서 시작한 코드 구문을 닫습니다.
)
# 심사 항목별 통과·대기·실패 상태가 담긴 JSON 결과를 읽습니다.
readiness = json.loads((readiness_dir / "assessment_readiness.json").read_text(encoding="utf-8"))
# 현재 모드의 최종 심사 준비 상태를 출력합니다.
print("Assessment readiness:", readiness["overall_status"])
# 결과를 Colab 표 형태로 표시합니다.
display(
    # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
    pd.DataFrame(readiness["checks"])[["criterion", "check_id", "status", "message", "evidence"]]
# 앞에서 시작한 코드 구문을 닫습니다.
)

## 12. 생성 결과 위치 요약

In [ ]:
# 최종 확인할 필수 표·보고서·선택 기록·백데이터 목록을 정의합니다.
expected_outputs = {
    # Whisper 모델 후보 비교 CSV 경로를 등록합니다.
    "benchmark_table": benchmark_dir / "benchmark_comparison.csv",
    # 모델 비교 결과 Word 보고서 경로를 등록합니다.
    "benchmark_report": benchmark_dir / "reports/benchmark_report.docx",
    # 선택 모델과 근거를 기록한 YAML 경로를 등록합니다.
    "model_selection": benchmark_dir / "model_selection.yaml",
    # 선택 모델 LoRA 학습·비교 결과 JSON 경로를 등록합니다.
    "selected_training": benchmark_dir / "selected_training_result.json",
    # 양자화 후보 비교 CSV 경로를 등록합니다.
    "quantization_table": quantization_dir / "quantization_comparison.csv",
    # 양자화 비교 결과 Word 보고서 경로를 등록합니다.
    "quantization_report": quantization_dir / "reports/quantization_report.docx",
    # 선택 양자화와 근거를 기록한 YAML 경로를 등록합니다.
    "quantization_selection": quantization_dir / "quantization_selection.yaml",
    # 선택 완료 후 고정 Test 결과 JSON 경로를 등록합니다.
    "final_test": quantization_dir / "final_test_result.json",
    # 심사 준비도 기계 판독용 JSON 경로를 등록합니다.
    "readiness_json": readiness_dir / "assessment_readiness.json",
    # 심사 준비도 사람이 읽을 Markdown 경로를 등록합니다.
    "readiness_markdown": readiness_dir / "assessment_readiness.md",
    # 모든 실험 이력을 누적한 SQLite 백데이터 경로를 등록합니다.
    "experiment_database": Path(DRIVE_ROOT) / "backdata/experiments.sqlite3",
# 앞에서 시작한 코드 구문을 닫습니다.
}
# 결과를 Colab 표 형태로 표시합니다.
display(
    # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
    pd.DataFrame(
        # 여러 값으로 구성된 자료 구조를 시작합니다.
        [
            # 여러 값으로 구성된 자료 구조를 시작합니다.
            {"artifact": name, "exists": path.exists(), "path": str(path)}
            # 각 항목을 순회하며 같은 처리를 반복합니다.
            for name, path in expected_outputs.items()
        # 앞에서 시작한 코드 구문을 닫습니다.
        ]
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
# 앞에서 시작한 코드 구문을 닫습니다.
)

# 공개·합성 기능 검증 모드이면 순위 기반 자동 선택을 사용합니다.
if IS_AUTOMATED_PROXY:
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print(
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        "다음 단계: DATA_MODE을 PRIVATE_MANUFACTURING으로 바꾸고 승인된 제조 "
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        "녹음·정답 전사를 넣은 뒤 같은 순서를 다시 실행합니다."
    # 앞에서 시작한 코드 구문을 닫습니다.
    )
# 앞선 조건에 해당하지 않는 경우를 처리합니다.
else:
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print(
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        "최종 보고서와 오류 샘플을 사람이 검수하고 human_review_signoff.yaml을 "
        # 이 줄의 연산 결과를 현재 데이터 준비·평가·선택 단계에 적용합니다.
        "완료한 뒤 assessment-audit --fail-on-blocker로 최종 확인하세요."
    # 앞에서 시작한 코드 구문을 닫습니다.
    )